# Weld Defect Analyzer — v2 (Preprocessed) GPU Training — **Kaggle**

Trains `weld_v2_preprocessed` (YOLOv8n-cls) on RIAWELC on a free Kaggle GPU (P100 / T4 x2).
Full 50 epochs in ~1–2 hours instead of ~71 hours on CPU. No Google Drive needed.

### One-time setup
1. **Get the data onto Kaggle** (pick one):
   - Search Datasets for **"RIAWELC"** — if a public version exists with `training/validation/testing` × `Difetto1/2/4/NoDifetto`, click **+ Add Data** to attach it (no upload).
   - Or create your own: **Datasets → New Dataset → Upload** `DB - Copy.zip` (Kaggle auto-extracts it), then **+ Add Data** to attach it to this notebook.
2. **Notebook settings (right panel):** Accelerator → **GPU T4 x2** (or P100); Internet → **On** (needed to pip-install ultralytics).
3. Run cells top to bottom. The trained model lands in `/kaggle/working/runs/...` — download it from the **Output** tab.

Then unzip into the repo at `runs/classify/weld_v2_preprocessed/weights/best.pt` so the
Streamlit app's *Preprocessed Model (weld_v2)* selector works.

In [ ]:
# 1) Verify GPU + install deps (requires Internet = On in the right panel)
import subprocess, torch
print('Torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU. Set Accelerator = GPU in the right panel, then re-run.')
subprocess.run(['pip', 'install', '-q', 'ultralytics>=8.3', 'opencv-python-headless', 'tqdm'], check=True)
print('Deps installed.')

In [ ]:
# 2) Locate the attached dataset under /kaggle/input (no Drive mount)
import os
WORK         = '/kaggle/working/weld'
PROCESSED    = f'{WORK}/data/processed'
PREPROCESSED = f'{WORK}/data/preprocessed'
RUN_PROJECT  = f'{WORK}/runs/classify'
RUN_NAME     = 'weld_v2_preprocessed'

# Find the folder that contains training/validation/testing inside /kaggle/input
RAW_BASE = None
for root, dirs, _ in os.walk('/kaggle/input'):
    low = {d.lower() for d in dirs}
    if {'training', 'validation', 'testing'}.issubset(low):
        RAW_BASE = root
        break
assert RAW_BASE, 'Dataset not found. Attach it via + Add Data so /kaggle/input has training/validation/testing.'
print('Raw dataset base:', RAW_BASE)

In [ ]:
# 3) Map RIAWELC -> YOLO classification layout (data/processed)
#    /kaggle/input is read-only, so we copy into /kaggle/working.
import os, shutil
SPLIT_MAP = {'training': 'train', 'validation': 'val', 'testing': 'test'}
CLASS_MAP = {'Difetto1': 'CR', 'Difetto2': 'LP', 'Difetto4': 'PO', 'NoDifetto': 'ND'}

shutil.rmtree(PROCESSED, ignore_errors=True)
total = 0
for raw_split, yolo_split in SPLIT_MAP.items():
    for raw_class, yolo_class in CLASS_MAP.items():
        src = os.path.join(RAW_BASE, raw_split, raw_class)
        dst = os.path.join(PROCESSED, yolo_split, yolo_class)
        if not os.path.isdir(src):
            print('[SKIP]', src); continue
        os.makedirs(dst, exist_ok=True)
        files = [f for f in os.listdir(src) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        for f in files:
            shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
        total += len(files)
        print(f'[OK] {raw_split}/{raw_class} -> {yolo_split}/{yolo_class} ({len(files)})')
print(f'\nMapped {total} images into {PROCESSED}')

In [ ]:
# 4) Preprocess: CLAHE + non-local-means denoise -> data/preprocessed (parallel)
import cv2, numpy as np, shutil
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

def _preprocess_one(in_path, rel, out_dir):
    try:
        out_path = Path(out_dir) / rel
        out_path.parent.mkdir(parents=True, exist_ok=True)
        img = cv2.imread(in_path)
        if img is None:
            return False
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img = cv2.cvtColor(clahe.apply(gray), cv2.COLOR_GRAY2BGR)
        img = cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)
        cv2.imwrite(str(out_path), img)
        return True
    except Exception as e:
        print('[ERR]', in_path, e); return False

src_root = Path(PROCESSED)
files = [p for p in src_root.rglob('*') if p.suffix.lower() in ('.png', '.jpg', '.jpeg')]
print(f'Preprocessing {len(files)} images (CLAHE + denoise)...')
shutil.rmtree(PREPROCESSED, ignore_errors=True)
ok = 0
with ProcessPoolExecutor() as ex:
    futs = [ex.submit(_preprocess_one, str(p), str(p.relative_to(src_root)), PREPROCESSED) for p in files]
    for i, f in enumerate(as_completed(futs), 1):
        ok += bool(f.result())
        if i % 2000 == 0 or i == len(files):
            print(f'  {i}/{len(files)} (ok={ok})')
print(f'Done. {ok}/{len(files)} preprocessed -> {PREPROCESSED}')

In [ ]:
# 5) Train weld_v2_preprocessed on GPU (50 epochs)
from ultralytics import YOLO
model = YOLO('yolov8n-cls.pt')
results = model.train(
    data     = PREPROCESSED,
    epochs   = 50,
    imgsz    = 224,
    batch    = 64,
    device   = 0,
    project  = RUN_PROJECT,
    name     = RUN_NAME,
    patience = 10,
    plots    = True,
    save     = True,
)
print('\nTraining complete. Best:', f'{RUN_PROJECT}/{RUN_NAME}/weights/best.pt')

In [ ]:
# 6) Validate on the held-out test split + copy results to /kaggle/working root for easy download
import shutil
run_dir = f'{RUN_PROJECT}/{RUN_NAME}'
best = f'{run_dir}/weights/best.pt'

metrics = YOLO(best).val(data=PREPROCESSED, split='test')
print('Test top-1 acc:', getattr(metrics, 'top1', 'n/a'))

# Kaggle exposes everything under /kaggle/working in the Output tab.
shutil.make_archive('/kaggle/working/weld_v2_best', 'zip', run_dir)
print('Download weld_v2_best.zip from the Output tab (right panel).')

## Put the trained model back into the repo

Download `weld_v2_best.zip` from the **Output** tab, unzip it, and place its contents so the repo has:

```
runs/classify/weld_v2_preprocessed/
    weights/best.pt          <- required by the app
    results.csv
    confusion_matrix.png ...
```

`load_predictor` in `src/app.py` already looks for `runs/classify/weld_v2_preprocessed/weights/best.pt`,
so the **Preprocessed Model (weld_v2)** option works as soon as the file is there.

> `runs/` and `*.pt` are gitignored. To deploy v2 on Streamlit Cloud, force-add the weight:
> `git add -f runs/classify/weld_v2_preprocessed/weights/best.pt`.